In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install seaborn

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from itertools import product
import warnings

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, setup_colab_secrets

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════

TICKER = 'BTC-USD'
START_DATE = '2018-01-01'

# Default params from TradingView Pine Script "LT MA Cross"
FAST_LENGTH = 10
SLOW_LENGTH = 64
LEVERAGE = 1.0

# Backtest settings (standard)
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
FREQ = 'D'
TRAIN_RATIO = 0.60

# Download data
stock_data = download(TICKER, START_DATE)

if not stock_data.empty:
    print(f'Downloaded {len(stock_data)} bars for {TICKER}')
    print(f'Range: {stock_data.index[0].date()} to {stock_data.index[-1].date()}')
    print(stock_data.head())
else:
    print('Download failed')

In [ ]:
# TECHNICAL INDICATORS

if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[('Close', TICKER)].astype(float).squeeze()
    high = stock_data[('High', TICKER)].astype(float).squeeze()
    low = stock_data[('Low', TICKER)].astype(float).squeeze()
else:
    close = stock_data['Close'].astype(float).squeeze()
    high = stock_data['High'].astype(float).squeeze()
    low = stock_data['Low'].astype(float).squeeze()

close.name = 'price'

# Compute EMA indicators for display
ema_10 = pd.Series(talib.EMA(close.values, timeperiod=10), index=close.index, name='EMA_10')
ema_64 = pd.Series(talib.EMA(close.values, timeperiod=64), index=close.index, name='EMA_64')
sma_200 = pd.Series(talib.SMA(close.values, timeperiod=200), index=close.index, name='SMA_200')
rsi_14 = pd.Series(talib.RSI(close.values, timeperiod=14), index=close.index, name='RSI_14')

indicators_df = pd.DataFrame({
    'Close': close, 'EMA_10': ema_10, 'EMA_64': ema_64,
    'SMA_200': sma_200, 'RSI_14': rsi_14
})
print('Indicators computed:')
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES + IS/OOS SPLIT

def select_close_series(data, ticker):
    if isinstance(data.columns, pd.MultiIndex):
        s = data[('Close', ticker)].astype(float).squeeze()
    else:
        s = data['Close'].astype(float).squeeze()
    s.name = 'price'
    return s

full_close = select_close_series(stock_data, TICKER)
split_idx = int(len(full_close) * TRAIN_RATIO)
train_close = full_close.iloc[:split_idx].copy()
val_close = full_close.iloc[split_idx:].copy()

print(f'Full sample:  {len(full_close)} bars ({full_close.index[0].date()} to {full_close.index[-1].date()})')
print(f'In-sample:    {len(train_close)} bars ({train_close.index[0].date()} to {train_close.index[-1].date()})')
print(f'Out-of-sample: {len(val_close)} bars ({val_close.index[0].date()} to {val_close.index[-1].date()})')

## LT MA Cross — EMA Crossover Strategy

Converted from TradingView Pine Script. Long-only EMA crossover:

- **BUY**: Fast EMA crosses above Slow EMA
- **SELL**: Fast EMA crosses below Slow EMA
- **1-bar execution delay** on all signals (no lookahead)
- **100% of equity** invested per trade (default leverage = 1x)

Default params: Fast=10, Slow=64 (from Rick's TradingView script).

### Grid Search
- `fast_length`: [5, 8, 10, 12, 15, 20, 25, 30]
- `slow_length`: [30, 40, 50, 64, 80, 100, 120, 150, 200]

In [ ]:
# DEFINE PARAMETER RANGES

fast_range = [5, 8, 10, 12, 15, 20, 25, 30]
slow_range = [30, 40, 50, 64, 80, 100, 120, 150, 200]

# Filter: fast must be < slow
combos = [(f, s) for f, s in product(fast_range, slow_range) if f < s]

print(f'Fast lengths:  {fast_range}')
print(f'Slow lengths:  {slow_range}')
print(f'Total combos:  {len(combos)}')
print(f'First 10: {combos[:10]}')

In [ ]:
# INITIALIZE RESULTS COLLECTION

grid_search_results = []

METRIC_LIST = [
    'fast_length', 'slow_length',
    'sharpe_ratio', 'sortino_ratio', 'total_return', 'annualized_return',
    'max_drawdown', 'annualized_volatility', 'calmar_ratio',
    'total_trades', 'win_rate', 'profit_factor', 'expectancy',
    'avg_win', 'avg_loss', 'largest_win', 'largest_loss',
    'avg_trade_duration'
]
print(f'Tracking {len(METRIC_LIST)} metrics per combo')

In [ ]:
# EMA CROSSOVER GRID SEARCH ON TRAINING DATA

print(f'Running grid search on {len(combos)} combinations...')
print(f'Training set: {train_close.index[0].date()} to {train_close.index[-1].date()}')
print()

for i, (fast_len, slow_len) in enumerate(combos):
    if (i + 1) % 20 == 0:
        print(f'\U0001f504 Progress: {i+1}/{len(combos)}')

    fast_ema = pd.Series(talib.EMA(train_close.values, timeperiod=fast_len), index=train_close.index)
    slow_ema = pd.Series(talib.EMA(train_close.values, timeperiod=slow_len), index=train_close.index)

    # Signal logic: EMA crossover with 1-bar execution delay
    entries_raw = (fast_ema.shift(1) <= slow_ema.shift(1)) & (fast_ema > slow_ema)
    exits_raw = (fast_ema.shift(1) >= slow_ema.shift(1)) & (fast_ema < slow_ema)
    entries = entries_raw.shift(1).fillna(False)
    exits = exits_raw.shift(1).fillna(False)

    try:
        pf = vbt.Portfolio.from_signals(
            close=train_close, entries=entries, exits=exits,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
        )
        trades = pf.trades
        n_trades = trades.count()
        if n_trades < 5:
            continue

        tr = np.asarray(trades.returns.values if hasattr(trades.returns, 'values') else trades.returns).ravel()
        pos = tr[tr > 0]
        neg = tr[tr < 0]
        win_rate = (len(pos) / len(tr) * 100) if len(tr) > 0 else 0
        pf_ratio = pos.sum() / abs(neg.sum()) if len(neg) > 0 and abs(neg.sum()) > 0 else np.inf

        ann_ret = float(pf.annualized_return(freq=FREQ))
        max_dd = float(pf.max_drawdown())

        grid_search_results.append({
            'fast_length': fast_len,
            'slow_length': slow_len,
            'sharpe_ratio': float(pf.sharpe_ratio(freq=FREQ)),
            'sortino_ratio': float(pf.sortino_ratio(freq=FREQ)),
            'total_return': float(pf.total_return()),
            'annualized_return': ann_ret,
            'max_drawdown': max_dd,
            'annualized_volatility': float(pf.annualized_volatility(freq=FREQ)),
            'calmar_ratio': ann_ret / abs(max_dd) if abs(max_dd) > 1e-9 else np.nan,
            'total_trades': int(n_trades),
            'win_rate': win_rate,
            'profit_factor': pf_ratio,
            'expectancy': float(tr.mean()) if len(tr) > 0 else 0,
            'avg_win': float(pos.mean()) if len(pos) > 0 else 0,
            'avg_loss': float(neg.mean()) if len(neg) > 0 else 0,
            'largest_win': float(pos.max()) if len(pos) > 0 else 0,
            'largest_loss': float(neg.min()) if len(neg) > 0 else 0,
        })
    except Exception as e:
        if i < 3:
            print(f'⚠️ Error on combo ({fast_len},{slow_len}): {type(e).__name__}: {e}')
        continue

if len(grid_search_results) == 0:
    print('\n⚠️ No valid combos found! Check vectorbt version or data.')
    results_df = pd.DataFrame(columns=['fast_length', 'slow_length', 'sharpe_ratio',
                                        'total_return', 'max_drawdown', 'total_trades',
                                        'win_rate', 'profit_factor'])
else:
    results_df = pd.DataFrame(grid_search_results).sort_values('sharpe_ratio', ascending=False)
    print(f'\nCompleted: {len(results_df)} valid combos')
    print(f'\nTop 10 by Sharpe (IS):')
    print(results_df[['fast_length', 'slow_length', 'sharpe_ratio', 'total_return',
                       'max_drawdown', 'total_trades', 'win_rate', 'profit_factor']].head(10).to_string(index=False))

In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & COMPARISON TABLE

if results_df.empty:
    print('No results to validate.')
else:
    top5 = results_df.head(5)
    oos_results = []

    for _, row in top5.iterrows():
        fast_len = int(row['fast_length'])
        slow_len = int(row['slow_length'])

        # OOS signals
        fast_ema = pd.Series(talib.EMA(val_close.values, timeperiod=fast_len), index=val_close.index)
        slow_ema = pd.Series(talib.EMA(val_close.values, timeperiod=slow_len), index=val_close.index)
        entries_raw = (fast_ema.shift(1) <= slow_ema.shift(1)) & (fast_ema > slow_ema)
        exits_raw = (fast_ema.shift(1) >= slow_ema.shift(1)) & (fast_ema < slow_ema)
        entries = entries_raw.shift(1).fillna(False)
        exits = exits_raw.shift(1).fillna(False)

        pf_oos = vbt.Portfolio.from_signals(
            close=val_close, entries=entries, exits=exits,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
        )

        oos_results.append({
            'fast_length': fast_len,
            'slow_length': slow_len,
            'IS_sharpe': row['sharpe_ratio'],
            'IS_return': row['total_return'],
            'IS_max_dd': row['max_drawdown'],
            'IS_trades': row['total_trades'],
            'OOS_sharpe': float(pf_oos.sharpe_ratio(freq=FREQ)),
            'OOS_return': float(pf_oos.total_return()),
            'OOS_max_dd': float(pf_oos.max_drawdown()),
            'OOS_trades': int(pf_oos.trades.count()),
            'sharpe_decay': float(pf_oos.sharpe_ratio(freq=FREQ)) - row['sharpe_ratio'],
        })

    oos_df = pd.DataFrame(oos_results)
    print('IS vs OOS Comparison (Top 5 by IS Sharpe):')
    print('=' * 100)
    for _, r in oos_df.iterrows():
        decay_pct = (r['sharpe_decay'] / r['IS_sharpe'] * 100) if r['IS_sharpe'] != 0 else 0
        flag = '\u2705' if r['OOS_sharpe'] > 0.5 else '\u26a0\ufe0f' if r['OOS_sharpe'] > 0 else '\u274c'
        print(f"{flag} EMA({int(r['fast_length'])},{int(r['slow_length'])}) | "
              f"IS: Sharpe={r['IS_sharpe']:.3f} Ret={r['IS_return']:.2%} DD={r['IS_max_dd']:.2%} | "
              f"OOS: Sharpe={r['OOS_sharpe']:.3f} Ret={r['OOS_return']:.2%} DD={r['OOS_max_dd']:.2%} | "
              f"Decay: {decay_pct:+.0f}%")
    print('=' * 100)

    # Select best OOS params
    best_oos = oos_df.loc[oos_df['OOS_sharpe'].idxmax()]
    BEST_FAST = int(best_oos['fast_length'])
    BEST_SLOW = int(best_oos['slow_length'])
    print(f'\nBest OOS params: EMA({BEST_FAST}, {BEST_SLOW}) — OOS Sharpe: {best_oos["OOS_sharpe"]:.3f}')

    # Plot IS vs OOS equity for top params
    fig, axes = plt.subplots(1, min(5, len(oos_df)), figsize=(20, 5))
    if len(oos_df) == 1:
        axes = [axes]
    for idx, (_, r) in enumerate(oos_df.iterrows()):
        fl, sl = int(r['fast_length']), int(r['slow_length'])
        # Full sample backtest for equity curve
        fe = pd.Series(talib.EMA(full_close.values, timeperiod=fl), index=full_close.index)
        se = pd.Series(talib.EMA(full_close.values, timeperiod=sl), index=full_close.index)
        ent = ((fe.shift(1) <= se.shift(1)) & (fe > se)).shift(1).fillna(False)
        ext = ((fe.shift(1) >= se.shift(1)) & (fe < se)).shift(1).fillna(False)
        pf_full = vbt.Portfolio.from_signals(close=full_close, entries=ent, exits=ext,
                                            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
        eq = pf_full.value()
        axes[idx].plot(eq.index[:split_idx], eq.values[:split_idx], color='#1f77b4', linewidth=1.5, label='IS')
        axes[idx].plot(eq.index[split_idx:], eq.values[split_idx:], color='#ff7f0e', linewidth=1.5, label='OOS')
        axes[idx].axvline(x=full_close.index[split_idx], color='red', linestyle=':', alpha=0.5)
        axes[idx].set_title(f'EMA({fl},{sl})\nIS={r["IS_sharpe"]:.2f} OOS={r["OOS_sharpe"]:.2f}', fontsize=10)
        axes[idx].legend(fontsize=8)
        axes[idx].grid(True, alpha=0.2)
        axes[idx].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    plt.tight_layout()
    plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PERFORMANCE DASHBOARD — Full-sample evaluation with best OOS params
# ════════════════════════════════════════════════════════════════════════

# Use best OOS params (or defaults if grid search didn't run)
try:
    b_fast = BEST_FAST
    b_slow = BEST_SLOW
except NameError:
    b_fast = FAST_LENGTH
    b_slow = SLOW_LENGTH

# Compute signals on full sample
fast_ema_full = pd.Series(talib.EMA(full_close.values, timeperiod=b_fast), index=full_close.index)
slow_ema_full = pd.Series(talib.EMA(full_close.values, timeperiod=b_slow), index=full_close.index)

entries_raw = (fast_ema_full.shift(1) <= slow_ema_full.shift(1)) & (fast_ema_full > slow_ema_full)
exits_raw = (fast_ema_full.shift(1) >= slow_ema_full.shift(1)) & (fast_ema_full < slow_ema_full)
entries = entries_raw.shift(1).fillna(False)
exits = exits_raw.shift(1).fillna(False)

# Strategy portfolio
pf = vbt.Portfolio.from_signals(
    close=full_close, entries=entries, exits=exits,
    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
)

# Buy & Hold portfolio
bh_entries = pd.Series(False, index=full_close.index, dtype=bool)
bh_entries.iloc[0] = True
bh_exits = pd.Series(False, index=full_close.index, dtype=bool)
pf_bh = vbt.Portfolio.from_signals(
    close=full_close, entries=bh_entries, exits=bh_exits,
    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ
)

# ── Compute all metrics ──
strat_ret = float(pf.total_return())
strat_ann_ret = float(pf.annualized_return(freq=FREQ))
strat_sharpe = float(pf.sharpe_ratio(freq=FREQ))
strat_sortino = float(pf.sortino_ratio(freq=FREQ))
strat_maxdd = float(pf.max_drawdown())
strat_vol = float(pf.annualized_volatility(freq=FREQ))
strat_calmar = strat_ann_ret / abs(strat_maxdd) if abs(strat_maxdd) > 1e-9 else np.nan
strat_trades = len(pf.trades)

trades_obj = pf.trades
tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
pos = tr[tr > 0]
neg = tr[tr < 0]
win_rate = (len(pos) / len(tr) * 100) if len(tr) > 0 else 0
pf_ratio = pos.sum() / abs(neg.sum()) if len(neg) > 0 and abs(neg.sum()) > 0 else np.inf

bh_ret = float(pf_bh.total_return())
bh_sharpe = float(pf_bh.sharpe_ratio(freq=FREQ))
bh_maxdd = float(pf_bh.max_drawdown())

# Print comparison table
print('=' * 70)
print(f'FULL-SAMPLE EVALUATION: EMA(fast={b_fast}, slow={b_slow})')
print(f'Period: {full_close.index[0].date()} to {full_close.index[-1].date()}')
print('=' * 70)
print(f'{"Metric":<28} {"Strategy":>14} {"Buy & Hold":>14}')
print('-' * 56)
print(f'{"Total Return":<28} {strat_ret:>13.2%} {bh_ret:>13.2%}')
print(f'{"Annualized Return":<28} {strat_ann_ret:>13.2%} {float(pf_bh.annualized_return(freq=FREQ)):>13.2%}')
print(f'{"Sharpe Ratio":<28} {strat_sharpe:>14.3f} {bh_sharpe:>14.3f}')
print(f'{"Sortino Ratio":<28} {strat_sortino:>14.3f} {float(pf_bh.sortino_ratio(freq=FREQ)):>14.3f}')
print(f'{"Max Drawdown":<28} {strat_maxdd:>13.2%} {bh_maxdd:>13.2%}')
print(f'{"Win Rate":<28} {win_rate:>13.1f}% {"N/A":>14}')
print(f'{"Profit Factor":<28} {pf_ratio:>14.2f} {"N/A":>14}')
print(f'{"Total Trades":<28} {strat_trades:>14d} {"1 (hold)":>14}')
print('=' * 70)

# ════════════════════════════════════════════════════════════════════════
# PERFORMANCE DASHBOARD FIGURE
# ════════════════════════════════════════════════════════════════════════

returns = pf.returns()
cum_returns = (1 + returns).cumprod() - 1
equity = pf.value()
running_max = equity.cummax()
drawdown = (equity - running_max) / running_max * 100

# Monthly returns
monthly_rets = returns.resample('M').apply(lambda x: (1 + x).prod() - 1) * 100
monthly_data = pd.DataFrame({
    'year': monthly_rets.index.year,
    'month': monthly_rets.index.month,
    'return': monthly_rets.values
})
pivot = monthly_data.pivot_table(values='return', index='month', columns='year')

# Yearly returns
yearly_rets = returns.resample('YE').apply(lambda x: (1 + x).prod() - 1) * 100

# Rolling metrics
rolling_sharpe = returns.rolling(252).mean() / returns.rolling(252).std() * np.sqrt(252)
rolling_vol = returns.rolling(252).std() * np.sqrt(252) * 100

# Buy & Hold cumulative
bh_cum = (1 + pf_bh.returns()).cumprod() - 1

# ── Build the dashboard ──
fig = plt.figure(figsize=(22, 28))
fig.suptitle(f'{TICKER} LT MA Cross — EMA({b_fast},{b_slow}) Performance Dashboard',
             fontsize=22, fontweight='bold', y=0.995)

gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.38, wspace=0.45,
                       top=0.97, bottom=0.03, left=0.06, right=0.97)

# ── Row 1: Cumulative Returns + Key Metrics ──
ax1 = fig.add_subplot(gs[0, :5])
ax1.fill_between(cum_returns.index, cum_returns.values * 100, alpha=0.25, color='#2ca02c')
ax1.plot(cum_returns.index, cum_returns.values * 100, color='#2ca02c', linewidth=1.8)
ax1.set_title('Cumulative Returns (Net)', fontsize=15, fontweight='bold')
ax1.set_ylabel('Return (%)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}%'))

# Key metrics box overlay
metrics_text = (
    f'KEY METRICS\n'
    f'Total Return: {strat_ret*100:>8.1f}%\n'
    f'CAGR:         {strat_ann_ret*100:>8.1f}%\n'
    f'Sharpe:       {strat_sharpe:>8.2f}\n'
    f'Sortino:      {strat_sortino:>8.2f}\n'
    f'Max DD:       {strat_maxdd*100:>7.1f}%\n'
    f'Win Rate:     {win_rate:>7.1f}%\n'
    f'Profit Factor:{pf_ratio:>7.2f}\n'
    f'Trades:       {strat_trades:>8d}'
)
ax_m = fig.add_subplot(gs[0, 5])
ax_m.axis('off')
ax_m.text(0.05, 0.95, metrics_text, transform=ax_m.transAxes,
          fontsize=12, verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='#333', alpha=0.95))

# ── Row 2: Drawdown + Monthly Heatmap ──
ax2 = fig.add_subplot(gs[1, :4])
ax2.fill_between(drawdown.index, drawdown.values, alpha=0.45, color='#8b0000')
ax2.plot(drawdown.index, drawdown.values, color='#8b0000', linewidth=0.8)
worst_idx = drawdown.idxmin()
ax2.scatter([worst_idx], [drawdown.min()], color='red', s=100, zorder=5, edgecolors='black')
ax2.set_title('Drawdown Over Time', fontsize=15, fontweight='bold')
ax2.set_ylabel('Drawdown (%)', fontsize=12)
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 4:])
month_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']
y_labels = [month_labels[m-1] for m in pivot.index]
sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
            ax=ax3, cbar_kws={'label': 'Return (%)', 'shrink': 0.8},
            yticklabels=y_labels, linewidths=0.5, linecolor='white')
ax3.set_title('Monthly Returns (%)', fontsize=15, fontweight='bold')
ax3.set_ylabel('month', fontsize=11)
ax3.set_xlabel('year', fontsize=11)

# ── Row 3: Strategy vs B&H + Yearly Results ──
ax4 = fig.add_subplot(gs[2, :3])
common_idx = cum_returns.index.intersection(bh_cum.index)
ax4.plot(common_idx, cum_returns.loc[common_idx].values * 100, color='#2ca02c',
         linewidth=1.8, label='Strategy')
ax4.plot(common_idx, bh_cum.loc[common_idx].values * 100, color='gray',
         linewidth=1.5, alpha=0.7, label='Buy & Hold', linestyle='--')
ax4.fill_between(common_idx, cum_returns.loc[common_idx].values * 100, alpha=0.12, color='#2ca02c')
ax4.axvline(x=full_close.index[split_idx], color='red', linestyle=':', alpha=0.5, label='IS/OOS Split')
ax4.set_title('Strategy vs Buy & Hold', fontsize=15, fontweight='bold')
ax4.set_ylabel('Return (%)', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[2, 3:])
years = yearly_rets.index.year
colors_yr = ['#2ca02c' if v >= 0 else '#d62728' for v in yearly_rets.values]
ax5.bar(years, yearly_rets.values, color=colors_yr, alpha=0.85, edgecolor='darkgreen', linewidth=0.5)
if len(yearly_rets) > 1:
    complete_years = yearly_rets.iloc[:-1]
    mean_ret = complete_years.mean()
    ax5.axhline(y=mean_ret, color='blue', linestyle='--', linewidth=1.8,
                label=f'Mean (complete years): {mean_ret:.1f}%')
    ax5.legend(fontsize=10)
ax5.set_title('Yearly Results', fontsize=15, fontweight='bold')
ax5.set_ylabel('Return (%)', fontsize=12)
ax5.grid(True, alpha=0.3, axis='y')

# ── Row 4: Returns Distribution + Rolling Sharpe + Rolling Vol ──
ax6 = fig.add_subplot(gs[3, :2])
daily_rets_pct = returns.dropna() * 100
ax6.hist(daily_rets_pct, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white', linewidth=0.3)
x_norm = np.linspace(daily_rets_pct.min(), daily_rets_pct.max(), 100)
ax6.plot(x_norm, stats.norm.pdf(x_norm, daily_rets_pct.mean(), daily_rets_pct.std()),
         'r--', linewidth=2, label='Normal')
ax6.set_title('Daily Returns Distribution', fontsize=15, fontweight='bold')
ax6.set_xlabel('Daily Return (%)', fontsize=12)
ax6.set_ylabel('Density', fontsize=12)
ax6.legend(fontsize=11)
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[3, 2:4])
ax7.plot(rolling_sharpe.index, rolling_sharpe.values, color='purple', linewidth=1)
mean_rs = rolling_sharpe.dropna().mean()
ax7.axhline(y=mean_rs, color='blue', linestyle='--', linewidth=1.8,
            label=f'Mean: {mean_rs:.2f}')
ax7.axhline(y=0, color='black', linewidth=0.5, alpha=0.3)
ax7.set_title('Rolling Sharpe (252 days)', fontsize=15, fontweight='bold')
ax7.set_ylabel('Sharpe Ratio', fontsize=12)
ax7.legend(fontsize=11)
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[3, 4:])
ax8.plot(rolling_vol.index, rolling_vol.values, color='#ff8c00', linewidth=1)
ax8.set_title('Rolling Volatility (252 days)', fontsize=15, fontweight='bold')
ax8.set_ylabel('Annualized Volatility (%)', fontsize=12)
ax8.grid(True, alpha=0.3)

# ── Row 5: Trade-by-trade analysis (3 panels) ──
trade_pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
cum_pnl = np.cumsum(trade_pnl)

ax9 = fig.add_subplot(gs[4, :2])
t_colors = ['#2ca02c' if r > 0 else '#d62728' for r in tr]
ax9.bar(range(len(tr)), tr * 100, color=t_colors, edgecolor='black', linewidth=0.3)
ax9.axhline(0, color='black', linewidth=1)
ax9.axhline(np.mean(tr) * 100, color='blue', linestyle='--', linewidth=1.5,
            label=f'Avg: {np.mean(tr)*100:.2f}%')
ax9.set_title('Individual Trade Returns (%)', fontsize=15, fontweight='bold')
ax9.set_xlabel('Trade #', fontsize=12)
ax9.set_ylabel('Return %', fontsize=12)
ax9.legend(fontsize=10)
ax9.grid(axis='y', alpha=0.3)

ax10 = fig.add_subplot(gs[4, 2:4])
ax10.plot(range(1, len(cum_pnl) + 1), cum_pnl, color='#1f77b4', linewidth=2, marker='o', markersize=3)
ax10.fill_between(range(1, len(cum_pnl) + 1), cum_pnl, 0,
                  where=cum_pnl >= 0, alpha=0.15, color='green')
ax10.fill_between(range(1, len(cum_pnl) + 1), cum_pnl, 0,
                  where=cum_pnl < 0, alpha=0.15, color='red')
ax10.axhline(0, color='black', linewidth=1)
ax10.set_title('Cumulative Trade P&L', fontsize=15, fontweight='bold')
ax10.set_xlabel('Trade #', fontsize=12)
ax10.set_ylabel('Cumulative P&L ($)', fontsize=12)
ax10.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax10.grid(True, alpha=0.3)

ax11 = fig.add_subplot(gs[4, 4:])
ax11.hist(tr * 100, bins=min(30, max(10, len(tr) // 3)),
          color='#1f77b4', edgecolor='black', alpha=0.7)
ax11.axvline(0, color='black', linewidth=1.5)
ax11.axvline(np.mean(tr) * 100, color='red', linestyle='--', linewidth=2,
             label=f'Mean: {np.mean(tr)*100:.2f}%')
ax11.axvline(np.median(tr) * 100, color='orange', linestyle='--', linewidth=2,
             label=f'Median: {np.median(tr)*100:.2f}%')
ax11.set_title('Trade Return Distribution', fontsize=15, fontweight='bold')
ax11.set_xlabel('Return %', fontsize=12)
ax11.set_ylabel('Frequency', fontsize=12)
ax11.legend(fontsize=10)
ax11.grid(axis='y', alpha=0.3)

plt.show()

# Save dashboard figure for export
dashboard_fig = fig
print(f'\nDashboard generated for EMA({b_fast},{b_slow}) on {TICKER}')

In [ ]:
# MONTE CARLO SIMULATION — FTMO CHALLENGE PASS PROBABILITY

print('=' * 70)
print('\U0001f3b2 MONTE CARLO SIMULATION — FTMO CHALLENGE FEASIBILITY')
print('=' * 70)

# Get daily returns of the strategy
daily_rets = pf.returns().values.ravel()
daily_rets = daily_rets[~np.isnan(daily_rets)]

# FTMO Parameters
FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
TRADING_DAYS = 30
N_SIMULATIONS = 10_000

print(f'\nStrategy: EMA(fast={b_fast}, slow={b_slow})')
print(f'Observed daily returns: {len(daily_rets)} days')
print(f'  Mean daily return:    {np.mean(daily_rets)*100:.4f}%')
print(f'  Std daily return:     {np.std(daily_rets)*100:.4f}%')
print(f'  Skewness:             {float(stats.skew(daily_rets)):.3f}')
print(f'  Kurtosis:             {float(stats.kurtosis(daily_rets)):.3f}')
print(f'\nFTMO Challenge Rules:')
print(f'  Account Size:         ${FTMO_ACCOUNT:,.0f}')
print(f'  Profit Target:        {PROFIT_TARGET:.0%} (${FTMO_ACCOUNT * PROFIT_TARGET:,.0f})')
print(f'  Max Daily Loss:       {MAX_DAILY_LOSS:.0%} (${FTMO_ACCOUNT * MAX_DAILY_LOSS:,.0f})')
print(f'  Max Total Loss:       {MAX_TOTAL_LOSS:.0%} (${FTMO_ACCOUNT * MAX_TOTAL_LOSS:,.0f})')
print(f'  Simulation Window:    {TRADING_DAYS} trading days')
print(f'  Simulations:          {N_SIMULATIONS:,}')

# Run Monte Carlo
np.random.seed(42)
equity_paths = np.zeros((N_SIMULATIONS, TRADING_DAYS + 1))
equity_paths[:, 0] = FTMO_ACCOUNT

passed = np.zeros(N_SIMULATIONS, dtype=bool)
blown = np.zeros(N_SIMULATIONS, dtype=bool)
daily_blown = np.zeros(N_SIMULATIONS, dtype=bool)
final_equity = np.zeros(N_SIMULATIONS)

for sim in range(N_SIMULATIONS):
    sim_rets = np.random.choice(daily_rets, size=TRADING_DAYS, replace=True)
    eq = FTMO_ACCOUNT
    day_start_eq = FTMO_ACCOUNT

    for day in range(TRADING_DAYS):
        day_start_eq = eq
        eq = eq * (1 + sim_rets[day])
        equity_paths[sim, day + 1] = eq

        # Check daily loss
        daily_loss = (eq - day_start_eq) / FTMO_ACCOUNT
        if daily_loss < -MAX_DAILY_LOSS:
            daily_blown[sim] = True
            equity_paths[sim, day + 2:] = eq
            break

        # Check total loss
        total_loss = (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT
        if total_loss < -MAX_TOTAL_LOSS:
            blown[sim] = True
            equity_paths[sim, day + 2:] = eq
            break

        # Check profit target
        if total_loss >= PROFIT_TARGET:
            passed[sim] = True
            equity_paths[sim, day + 2:] = eq
            break

    final_equity[sim] = equity_paths[sim, -1]

n_passed = passed.sum()
n_blown_total = blown.sum()
n_blown_daily = daily_blown.sum()
n_survived = N_SIMULATIONS - n_blown_total - n_blown_daily - n_passed

print(f'\n{"=" * 70}')
print(f'\U0001f3af MONTE CARLO RESULTS ({N_SIMULATIONS:,} simulations)')
print(f'{"=" * 70}')
print(f'  \u2705 PASSED (hit {PROFIT_TARGET:.0%} target):    {n_passed:>6,} ({n_passed/N_SIMULATIONS*100:.1f}%)')
print(f'  \u274c BLOWN (max total loss):       {n_blown_total:>6,} ({n_blown_total/N_SIMULATIONS*100:.1f}%)')
print(f'  \u274c BLOWN (max daily loss):       {n_blown_daily:>6,} ({n_blown_daily/N_SIMULATIONS*100:.1f}%)')
print(f'  \u23f3 Still trading (no trigger):   {n_survived:>6,} ({n_survived/N_SIMULATIONS*100:.1f}%)')
print(f'  \U0001f4b0 Median final equity:         ${np.median(final_equity):>10,.0f}')
print(f'  \U0001f4b0 Mean final equity:           ${np.mean(final_equity):>10,.0f}')
print(f'  \U0001f4b0 5th percentile equity:       ${np.percentile(final_equity, 5):>10,.0f}')
print(f'  \U0001f4b0 95th percentile equity:      ${np.percentile(final_equity, 95):>10,.0f}')
print(f'{"=" * 70}')

# Monte Carlo Visualization
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(f'Monte Carlo FTMO Challenge — EMA({b_fast},{b_slow}) {TICKER} — {N_SIMULATIONS:,} Paths',
             fontsize=16, fontweight='bold')

# Equity paths
sample_idx = np.random.choice(N_SIMULATIONS, min(200, N_SIMULATIONS), replace=False)
for i in sample_idx:
    c = '#2ca02c' if passed[i] else ('#d62728' if (blown[i] or daily_blown[i]) else '#aaaaaa')
    a = 0.4 if (passed[i] or blown[i] or daily_blown[i]) else 0.1
    axes[0, 0].plot(range(TRADING_DAYS + 1), equity_paths[i], color=c, alpha=a, linewidth=0.5)
axes[0, 0].axhline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2.5,
                   label=f'Target (${FTMO_ACCOUNT*(1+PROFIT_TARGET):,.0f})')
axes[0, 0].axhline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2.5,
                   label=f'Max Loss (${FTMO_ACCOUNT*(1-MAX_TOTAL_LOSS):,.0f})')
axes[0, 0].set_title('Simulated Equity Paths', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Trading Day')
axes[0, 0].set_ylabel('Equity ($)')
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0, 0].legend(loc='upper left', fontsize=9)
axes[0, 0].grid(True, alpha=0.2)

# Final equity distribution
bins = np.linspace(final_equity.min(), final_equity.max(), 60)
axes[0, 1].hist(final_equity[passed], bins=bins, color='#2ca02c', alpha=0.7, label=f'Passed ({n_passed:,})')
axes[0, 1].hist(final_equity[blown | daily_blown], bins=bins, color='#d62728', alpha=0.7,
                label=f'Blown ({n_blown_total+n_blown_daily:,})')
axes[0, 1].hist(final_equity[~passed & ~blown & ~daily_blown], bins=bins, color='#aaaaaa', alpha=0.5,
                label=f'Still Trading ({n_survived:,})')
axes[0, 1].axvline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2)
axes[0, 1].axvline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2)
axes[0, 1].set_title('Final Equity Distribution', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Final Equity ($)')
axes[0, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.2)

# Outcome pie chart
labels = ['Passed', 'Blown (Total)', 'Blown (Daily)', 'Still Trading']
sizes = [n_passed, n_blown_total, n_blown_daily, n_survived]
colors_pie = ['#2ca02c', '#d62728', '#ff7f0e', '#aaaaaa']
explode = (0.05, 0.05, 0.05, 0)
non_zero = [(l, s, c, e) for l, s, c, e in zip(labels, sizes, colors_pie, explode) if s > 0]
if non_zero:
    labels_f, sizes_f, colors_f, explode_f = zip(*non_zero)
    axes[1, 0].pie(sizes_f, explode=explode_f, labels=labels_f, colors=colors_f,
                   autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 11})
    axes[1, 0].set_title('Outcome Distribution', fontsize=13, fontweight='bold')

# Percentile curves
pcts = [5, 25, 50, 75, 95]
pct_colors = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c', '#17becf']
for pct, clr in zip(pcts, pct_colors):
    pct_path = np.percentile(equity_paths, pct, axis=0)
    axes[1, 1].plot(range(TRADING_DAYS + 1), pct_path, color=clr, linewidth=2, label=f'P{pct}')
axes[1, 1].axhline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2, alpha=0.7)
axes[1, 1].axhline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2, alpha=0.7)
axes[1, 1].set_title('Percentile Equity Curves', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Trading Day')
axes[1, 1].set_ylabel('Equity ($)')
axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# FTMO Verdict
pass_rate = n_passed / N_SIMULATIONS * 100
mc_pass_rate = pass_rate
print(f'\n{"=" * 70}')
if pass_rate >= 50:
    print(f'\U0001f7e2 FTMO VERDICT: FAVORABLE — {pass_rate:.1f}% pass rate')
elif pass_rate >= 25:
    print(f'\U0001f7e1 FTMO VERDICT: POSSIBLE — {pass_rate:.1f}% pass rate')
elif pass_rate >= 10:
    print(f'\U0001f7e0 FTMO VERDICT: CHALLENGING — {pass_rate:.1f}% pass rate')
else:
    print(f'\U0001f534 FTMO VERDICT: UNLIKELY — {pass_rate:.1f}% pass rate')
print(f'{"=" * 70}')

In [ ]:
# ================================================================
# Universal Export Cell — PDF Tearsheet + Data Files
# ================================================================

STRATEGY_NAME = 'LT_MA_Cross'
PARAM_COLS = ['fast_length', 'slow_length']

import os
_lib_dir = 'lib' if os.path.isdir('lib') else os.path.join('..', 'lib')
exec(open(os.path.join(_lib_dir, 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())